# Qwen3-1.7B DPO — POC Triage Médical CHSA

Alignement par préférences (DPO) du modèle **Qwen3-1.7B SFT** pour améliorer la classification triage P1/P2/P3.

**Pipeline :** Qwen3-1.7B-Base → SFT (LoRA) ✅ → **DPO** → Endpoint vLLM  
**Repo :** [XavierCoulon/OC_P14_Finetunez_votre_propre_LLM](https://github.com/XavierCoulon/OC_P14_Finetunez_votre_propre_LLM)  
**Modèle SFT :** [XavierCoulon/qwen3-1.7b-chsa-sft-lora](https://huggingface.co/XavierCoulon/qwen3-1.7b-chsa-sft-lora)  
**Modèle DPO publié :** [XavierCoulon/qwen3-1.7b-chsa-dpo](https://huggingface.co/XavierCoulon/qwen3-1.7b-chsa-dpo)

> Exécuter sur Kaggle ou Google Colab avec GPU (T4 minimum).  
> Prérequis Secrets : `HF_TOKEN`, `WANDB_API_KEY`.

In [ ]:
%%capture
import os, re

env_keys = "".join(os.environ.keys())
ON_COLAB  = "COLAB_" in env_keys
ON_KAGGLE = "KAGGLE_" in env_keys

if ON_COLAB or ON_KAGGLE:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub==0.27.1" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
else:
    !pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install wandb "ragas==0.2.6" langchain-mistralai langchain-huggingface sentence-transformers

In [ ]:
def get_secret(name):
    """Lit un secret depuis Kaggle, Colab ou variable d'environnement."""
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    if ON_COLAB:
        from google.colab import userdata
        return userdata.get(name)
    return os.environ.get(name, "")

import wandb
wandb.login(key=get_secret('WANDB_API_KEY'))
wandb.init(project="chsa-dpo-qwen3", name=f"run-{__import__('datetime').datetime.now().strftime('%Y%m%d-%H%M')}")

In [ ]:
# transformers 4.56+ : list_repo_templates lève RemoteEntryNotFoundError sur les repos
# sans dossier additional_chat_templates. On patch les deux modules qui détiennent une référence.
import transformers.utils.hub as _tfhub
import transformers.tokenization_utils_base as _tbase

_orig_list_repo_templates = _tfhub.list_repo_templates
def _safe_list_repo_templates(*args, **kwargs):
    try:
        return _orig_list_repo_templates(*args, **kwargs)
    except Exception:
        return []

_tfhub.list_repo_templates = _safe_list_repo_templates
_tbase.list_repo_templates = _safe_list_repo_templates

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None
load_in_4bit = True

# Charge le modèle SFT (adapters LoRA)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "XavierCoulon/qwen3-1.7b-chsa-sft-lora-merged",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

### Data Prep DPO

Dataset de préférences `XavierCoulon/oc-p14-dataset` (config `dpo`) — 1 600 paires `chosen`/`rejected` issues de UltraMedical-Preference.

Format attendu par DPOTrainer : `prompt` / `chosen` / `rejected` en format ChatML Qwen3.

In [ ]:
import random
random.seed(42)

from datasets import load_dataset

dataset = load_dataset("XavierCoulon/oc-p14-dataset", "dpo", split="train")
print(f"Dataset DPO train : {len(dataset)} paires")

val_raw = load_dataset("XavierCoulon/oc-p14-dataset", "dpo", split="val")
print(f"Dataset DPO val   : {len(val_raw)} paires")
print(dataset.column_names)


In [ ]:
import json, random
random.seed(42)

def format_dpo(examples):
    prompts, chosens, rejecteds = [], [], []
    for prompt, chosen, rejected in zip(
        examples["prompt"], examples["chosen"], examples["rejected"]
    ):
        think_tag = "/think" if random.random() < 0.75 else "/no_think"
        p = (
            f"<|im_start|>user\n{think_tag}\n{prompt}<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
        # chosen/rejected sont des dicts {"role": "assistant", "content": "..."}
        if isinstance(chosen, str):
            chosen = json.loads(chosen)
        if isinstance(rejected, str):
            rejected = json.loads(rejected)
        prompts.append(p)
        chosens.append(chosen["content"] + tokenizer.eos_token)
        rejecteds.append(rejected["content"] + tokenizer.eos_token)
    return {"prompt": prompts, "chosen": chosens, "rejected": rejecteds}

dataset = dataset.map(format_dpo, batched=True, remove_columns=dataset.column_names)
val_dataset = val_raw.map(format_dpo, batched=True, remove_columns=val_raw.column_names)
print(f"Train formaté : {len(dataset)} paires")
print(f"Val formaté   : {len(val_dataset)} paires")
print("\nExemple prompt :")
print(dataset[0]["prompt"][:200])
print("\nChosen :")
print(dataset[0]["chosen"][:200])


### Entraînement DPO

DPOTrainer avec LoRA — 1 epoch sur les 1 600 paires. `beta=0.1` contrôle l'écart par rapport au modèle de référence (SFT).

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")

In [ ]:
from trl import DPOConfig, DPOTrainer

trainer = DPOTrainer(
    model = model,
    ref_model = None,   # None = utilise le modèle de base gelé (recommandé avec LoRA)
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = val_dataset,
    args = DPOConfig(
        beta = 0.02,              # 0.05→0.02 : rewards/chosen encore négatifs (-0.206) sur run-20260529-0623
        max_length = 1024,
        max_prompt_length = 512,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,  # batch effectif = 8
        num_train_epochs = 2,             # 1→2 : loss eval encore en descente à epoch 1.0, margin faible (0.277)
        learning_rate = 1e-5,             # 2e-5→1e-5 : grad norms entre 5-13, clipping actif en permanence
        lr_scheduler_type = "cosine",
        warmup_ratio = 0.1,
        optim = "adamw_8bit",
        seed = 42,
        output_dir = "outputs_dpo",
        report_to = "wandb",
        save_strategy = "steps",
        save_steps = 50,                  # 100→50 : seulement 2 points d'eval avec 200 steps
        save_total_limit = 2,
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = 50,                  # 100→50 : idem
    ),
)


In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"{trainer_stats.metrics['train_runtime']/60:.2f} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")

### Sauvegarde et push vers HuggingFace Hub

In [ ]:
HF_TOKEN  = get_secret('HF_TOKEN')
HF_REPO   = "XavierCoulon/qwen3-1.7b-chsa-dpo"

model.save_pretrained("qwen_dpo_lora")
tokenizer.save_pretrained("qwen_dpo_lora")

model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f"Modèle DPO publié : https://huggingface.co/{HF_REPO}")

### Fusion pour déploiement vLLM

vLLM nécessite un modèle complet (pas des adapters LoRA). Cette cellule fusionne les poids et publie le modèle merged.

In [ ]:
HF_REPO_MERGED = "XavierCoulon/qwen3-1.7b-chsa-dpo-merged"

model.push_to_hub_merged(
    HF_REPO_MERGED,
    tokenizer,
    save_method = "merged_16bit",
    token = HF_TOKEN,
)
print(f"Modèle merged publié : https://huggingface.co/{HF_REPO_MERGED}")
print("→ Utiliser ce modèle dans docker-compose.yml (MODEL_NAME)")

In [ ]:
# wandb.finish() déplacé après eval_clinique pour logger les métriques dans le même run

### Évaluation — eval_clinique (100 cas)

Évaluation RAGAS du modèle DPO sur 100 cas cliniques isolés (aucun overlap avec les données d'entraînement SFT ni DPO).

**Métriques :**
- **FactualCorrectness** : exactitude factuelle vs référence (LLM-as-judge — Mistral)
- **ResponseRelevancy** : la réponse adresse-t-elle la question ? (Mistral + embeddings)
- **SemanticSimilarity** : proximité sémantique avec la référence (embeddings MiniLM)

> Permet de comparer directement avec le score SFT et de mesurer le gain apporté par l'alignement DPO.

In [ ]:
from datasets import load_dataset as load_eval_dataset
import pandas as pd
import wandb, torch, time, sys, types

# Guard MISTRAL_API_KEY avant de lancer 30+ min d'inférence
_mistral_key = get_secret("MISTRAL_API_KEY")
if not _mistral_key:
    raise RuntimeError("MISTRAL_API_KEY absent — ajoute le secret Kaggle avant de relancer.")
print(f"✅ MISTRAL_API_KEY détectée ({len(_mistral_key)} chars)")

eval_dataset = load_eval_dataset("XavierCoulon/oc-p14-dataset", "eval_clinique", split="eval")
n_total = len(eval_dataset)

FastLanguageModel.for_inference(model)
torch.cuda.empty_cache()

results      = []
results_full = []

print(f"▶ eval_clinique — {n_total} cas à évaluer\n")
t_start = time.time()

try:
    # ── Boucle d'inférence ────────────────────────────────────────────────────
    for i, example in enumerate(eval_dataset):
        prompt = (
            f"<|im_start|>user\n/no_think\n{example['instruction']}<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        if inputs["input_ids"].shape[1] > max_seq_length - 256:
            inputs = {k: v[:, -(max_seq_length - 256):] for k, v in inputs.items()}
        outputs = model.generate(**inputs, max_new_tokens=256,
                                 temperature=0.6, top_p=0.95, top_k=20,
                                 repetition_penalty=1.2, use_cache=True)
        generated = tokenizer.batch_decode(outputs)[0]
        response = generated.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

        results.append({
            "source":      example.get('source', 'unknown'),
            "instruction": example['instruction'][:80] + "...",
            "reference":   example['response'][:120] + "...",
            "generated":   response[:120] + "...",
        })
        results_full.append({
            "run_name":    wandb.run.name,
            "model":       "dpo",
            "source":      example.get('source', 'unknown'),
            "instruction": example['instruction'],
            "reference":   example['response'],
            "generated":   response,
        })

        if (i + 1) % 10 == 0 or (i + 1) == n_total:
            elapsed = time.time() - t_start
            remaining = (elapsed / (i + 1)) * (n_total - i - 1)
            print(f"  [{i+1:3d}/{n_total}] écoulé : {elapsed/60:.1f} min | restant : {remaining/60:.1f} min")

    elapsed_total = time.time() - t_start
    print(f"\n✅ Inférence terminée — {n_total} cas  [{elapsed_total/60:.1f} min]")

    csv_path = f"eval_clinique_dpo_{wandb.run.name}.csv"
    pd.DataFrame(results_full).to_csv(csv_path, index=False)
    print(f"→ {csv_path} exporté")

    # ── RAGAS ─────────────────────────────────────────────────────────────────
    # Patch : langchain-community >= 0.3 a supprimé le module vertexai que ragas importe
    if "langchain_community.chat_models.vertexai" not in sys.modules:
        _stub = types.ModuleType("langchain_community.chat_models.vertexai")
        class _ChatVertexAI: pass
        _stub.ChatVertexAI = _ChatVertexAI
        sys.modules["langchain_community.chat_models.vertexai"] = _stub

    ragas_metrics = {}
    print("\n▶ RAGAS (Mistral + MiniLM)...")
    try:
        from ragas import EvaluationDataset, SingleTurnSample, evaluate
        from ragas.run_config import RunConfig
        from ragas.metrics import FactualCorrectness, ResponseRelevancy, AnswerSimilarity
        from ragas.llms import LangchainLLMWrapper
        from ragas.embeddings import LangchainEmbeddingsWrapper
        from langchain_mistralai import ChatMistralAI
        from langchain_huggingface import HuggingFaceEmbeddings

        ragas_llm = LangchainLLMWrapper(
            ChatMistralAI(model="mistral-small-latest", api_key=_mistral_key)
        )
        ragas_emb = LangchainEmbeddingsWrapper(
            HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        )
        ragas_samples = [
            SingleTurnSample(
                user_input=r["instruction"],
                response=r["generated"],
                reference=r["reference"],
            )
            for r in results_full
        ]
        ragas_result = evaluate(
            dataset=EvaluationDataset(samples=ragas_samples),
            metrics=[
                FactualCorrectness(llm=ragas_llm),
                ResponseRelevancy(llm=ragas_llm, embeddings=ragas_emb),
                AnswerSimilarity(embeddings=ragas_emb),
            ],
            run_config=RunConfig(max_workers=8, timeout=120, max_retries=5),
        )
        df_ragas = ragas_result.to_pandas()
        # Alignement défensif : RAGAS peut retourner moins de lignes si certains appels échouent
        df_ragas["source"] = [r["source"] for r in results_full[:len(df_ragas)]]

        for metric in ["factual_correctness", "answer_relevancy", "semantic_similarity"]:
            ragas_metrics[metric] = float(df_ragas[metric].mean())
            print(f"  {metric:30s} : {ragas_metrics[metric]:.3f}")

        print("\n--- RAGAS par source ---")
        print(df_ragas.groupby("source")[
            ["factual_correctness", "answer_relevancy", "semantic_similarity"]
        ].mean().round(3).to_string())
        ragas_metrics["df"] = df_ragas

    except Exception as e_ragas:
        print(f"⚠️ RAGAS échoué : {e_ragas}")

    # ── W&B log ───────────────────────────────────────────────────────────────
    log_payload = {
        "eval_clinique/n_examples": n_total,
        "eval_clinique/examples": wandb.Table(
            columns=["source", "instruction", "reference", "generated"],
            data=[[r["source"], r["instruction"], r["reference"], r["generated"]]
                  for r in results],
        ),
    }
    for metric in ["factual_correctness", "answer_relevancy", "semantic_similarity"]:
        if metric in ragas_metrics:
            log_payload[f"eval_clinique/ragas_{metric}"] = ragas_metrics[metric]
    if "df" in ragas_metrics:
        log_payload["eval_clinique/ragas_by_source"] = wandb.Table(
            dataframe=ragas_metrics["df"]
                .groupby("source")[["factual_correctness", "answer_relevancy", "semantic_similarity"]]
                .mean().reset_index().round(3)
        )
    wandb.log(log_payload)

except Exception as e:
    print(f"⚠️ Erreur eval_clinique à l'exemple {len(results)+1}/{n_total} : {e}")
    raise

finally:
    wandb.finish()
    print("W&B run terminé.")